# Shop Cash Uplift — Experiment Opportunity Sizing

Eligible US users active in the last 30 days, segmented by order history. Covers population sizing, post-order LTV & retention analysis, and sample size / power calculations for the Phase 1 experiment.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
from shared.gcp.bigquery import BigQuery
from shared.sql import read_sql

bq = BigQuery(notebook_slug="opportunity-sizing")

## 1. Experiment Population

Eligible US users active in the last 30 days (safe risk, not on blocklist), segmented by order history.

In [ ]:
query = read_sql(
    "queries/user_ltv_data.sql",
    params={"lookback_days": 30, "order_lookback_days": 365},
)
user_ltv_df = bq.query(query).to_dataframe()

count_query = """
SELECT COUNT(DISTINCT deduped_user_id) AS active_user_count
FROM `sdp-prd-shop-ml.mart.mart__shop_app__user_profile`
WHERE last_shop_app_activity.activity_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  AND (last_shop_app_user_geo.country_code IN ('US') OR country_code IN ('US'))
  AND risk_level IN ('safe/not rated', 'safe', 'low_risk')
  AND NOT is_on_giveaways_blocklist
  AND NOT is_shopifolk_user
"""
total_active_users = bq.query(count_query).to_dataframe()["active_user_count"].iloc[0]

print(f"{total_active_users:,} active users (30-day window)")
print(f"{len(user_ltv_df):,} with orders in last 365 days")

In [ ]:
users_with_orders = len(user_ltv_df)
users_zero_orders = total_active_users - users_with_orders

segments = pd.DataFrame({
    "segment": [
        "New buyers (0 orders)",
        "1 order (1→2 conversion)",
        "2+ orders (repeat buyers)",
    ],
    "user_count": [
        users_zero_orders,
        int((user_ltv_df["total_orders"] == 1).sum()),
        int((user_ltv_df["total_orders"] >= 2).sum()),
    ],
})
segments["pct_of_active"] = (segments["user_count"] / total_active_users * 100).round(1)

fig = px.bar(
    segments,
    x="segment",
    y="user_count",
    text=segments.apply(lambda r: f"{r['user_count']:,.0f} ({r['pct_of_active']}%)", axis=1),
    title="Active Users by Order History Segment",
    color="segment",
    color_discrete_sequence=["#88C0D0", "#5E81AC", "#B48EAD"],
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Users", xaxis_title="")
fig.show()

display(segments)

## 2. Post-Order LTV & Retention

365-day LTV after each order milestone, repeat rates, and time-to-next-order. Sampled from 1M eligible US users whose milestone occurred 365+ days ago.

In [ ]:
query = read_sql(
    "queries/ltv_post_nth_order.sql",
    params={"sample_size": 1000000, "max_milestone": 5},
)
milestone_df = bq.query(query).to_dataframe()
print(f"Loaded {len(milestone_df):,} user-milestone records")

In [ ]:
ltv_by_milestone = (
    milestone_df.groupby("milestone")
    .agg(
        users=("deduped_user_id", "nunique"),
        mean_ltv=("gmv_365d_post", "mean"),
        median_ltv=("gmv_365d_post", "median"),
        p25_ltv=("gmv_365d_post", lambda x: np.percentile(x, 25)),
        p75_ltv=("gmv_365d_post", lambda x: np.percentile(x, 75)),
        mean_orders=("orders_365d_post", "mean"),
    )
    .reset_index()
)
ltv_by_milestone["milestone_label"] = ltv_by_milestone["milestone"].apply(lambda m: f"After order #{m}")

fig = px.bar(
    ltv_by_milestone,
    x="milestone_label",
    y="mean_ltv",
    error_y=ltv_by_milestone["p75_ltv"] - ltv_by_milestone["mean_ltv"],
    error_y_minus=ltv_by_milestone["mean_ltv"] - ltv_by_milestone["p25_ltv"],
    text=ltv_by_milestone["mean_ltv"].apply(lambda v: f"${v:,.0f}"),
    title="365-Day LTV After Each Order Milestone",
    color_discrete_sequence=["#5E81AC"],
    labels={"mean_ltv": "Mean 365-Day GMV (USD)", "milestone_label": ""},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

display(ltv_by_milestone[["milestone_label", "users", "mean_ltv", "median_ltv", "p25_ltv", "p75_ltv", "mean_orders"]])

In [ ]:
cap = np.percentile(milestone_df.loc[milestone_df["gmv_365d_post"] > 0, "gmv_365d_post"], 95)
milestone_plot = milestone_df.copy()
milestone_plot["gmv_clipped"] = milestone_plot["gmv_365d_post"].clip(lower=0, upper=cap)
milestone_plot["milestone_label"] = milestone_plot["milestone"].apply(lambda m: f"After order #{m}")

sample_n = 20_000
sampled = milestone_plot.groupby("milestone", group_keys=False).apply(
    lambda g: g.sample(min(sample_n, len(g)), random_state=42)
)

fig = px.box(
    sampled,
    x="milestone_label",
    y="gmv_clipped",
    color="milestone_label",
    title="365-Day LTV Distribution by Order Milestone",
    labels={"gmv_clipped": "365-Day GMV (USD)", "milestone_label": ""},
    color_discrete_sequence=["#5E81AC", "#81A1C1", "#88C0D0", "#8FBCBB", "#A3BE8C"],
)
fig.update_layout(showlegend=False)
fig.update_yaxes(title="365-Day Post-Milestone GMV (USD, capped at 95th pctile)")
fig.show()

In [ ]:
repeat_stats = (
    milestone_df.groupby("milestone")
    .agg(
        total_users=("deduped_user_id", "nunique"),
        users_with_next=("next_order_at", "count"),
        median_days=("days_to_next_order", "median"),
    )
    .reset_index()
)
repeat_stats["repeat_rate"] = (repeat_stats["users_with_next"] / repeat_stats["total_users"] * 100).round(1)
repeat_stats["milestone_label"] = repeat_stats["milestone"].apply(lambda m: f"Order #{m} → #{m+1}")

fig_rate = px.bar(
    repeat_stats,
    x="milestone_label",
    y="repeat_rate",
    text=repeat_stats["repeat_rate"].apply(lambda v: f"{v:.1f}%"),
    title="Repeat Rate: % Who Make the Next Order",
    color_discrete_sequence=["#88C0D0"],
    labels={"repeat_rate": "Repeat Rate (%)", "milestone_label": ""},
)
fig_rate.update_traces(textposition="outside")
fig_rate.update_layout(showlegend=False)

fig_days = px.bar(
    repeat_stats.dropna(subset=["median_days"]),
    x="milestone_label",
    y="median_days",
    text=repeat_stats.dropna(subset=["median_days"])["median_days"].apply(lambda v: f"{v:.0f}d"),
    title="Median Days to Next Order",
    color_discrete_sequence=["#B48EAD"],
    labels={"median_days": "Median Days", "milestone_label": ""},
)
fig_days.update_traces(textposition="outside")
fig_days.update_layout(showlegend=False)

fig_rate.show()
fig_days.show()

display(repeat_stats[["milestone_label", "total_users", "users_with_next", "repeat_rate", "median_days"]])

In [ ]:
from IPython.display import Markdown

m1 = ltv_by_milestone[ltv_by_milestone["milestone"] == 1].iloc[0]
m2 = ltv_by_milestone[ltv_by_milestone["milestone"] == 2].iloc[0]
r1 = repeat_stats[repeat_stats["milestone"] == 1].iloc[0]
r2 = repeat_stats[repeat_stats["milestone"] == 2].iloc[0]

churn_rate = 100 - r1["repeat_rate"]
ltv_lift = m2["mean_ltv"] - m1["mean_ltv"]

display(Markdown(
    "## Key Findings\n\n"
    "### The 1→2 order transition shows the steepest drop-off and largest LTV gap.\n\n"
    f"**Retention cliff.** {churn_rate:.0f}% of first-time buyers never make a second "
    "purchase — the largest drop-off in the funnel. After order #2, repeat rates rise to "
    f"{r2['repeat_rate']:.0f}%+ and continue climbing at each subsequent milestone.\n\n"
    "**LTV step change.** Median 365-day LTV after a first order is **\\$0** — more "
    "than half of first-time buyers generate no additional revenue. After a second order, "
    f"median LTV rises to **\\${m2['median_ltv']:,.0f}**, and mean LTV increases from "
    f"**\\${m1['mean_ltv']:,.0f}** to **\\${m2['mean_ltv']:,.0f}** "
    f"(+\\${ltv_lift:,.0f} per user). Users who reach order #2 go on to place "
    f"**{m2['mean_orders']:.1f}** additional orders on average, vs. "
    f"**{m1['mean_orders']:.1f}** after the first.\n\n"
    f"**Wide intervention window.** The median gap between order #1 and #2 is "
    f"**{r1['median_days']:.0f} days** — roughly double the gap at later milestones — "
    "suggesting there is time for an incentive to reach users before they churn.\n\n"
    "### Implications for experiment design\n\n"
    "These patterns suggest that the 1→2 conversion segment may be at least as "
    "valuable as first-purchase acquisition for a Shop Cash intervention. "
    "**Broadening experiment eligibility beyond new buyers** would allow us to "
    "test this directly and measure relative dollar efficiency across segments."
))

## 3. Sample Size Calculator

How many users do we need per arm to detect a meaningful lift in purchase conversion?

| Step | What we do |
|------|------------|
| **1. Baseline** | Measure organic conversion among eligible users active in the 30 days before a reference date |
| **2. Window** | Count how many make *any* purchase in the next 7 / 10 / 14 days |
| **3. Power** | Calculate the sample size needed to detect a given % lift above that organic rate |

Eligibility filters match the planned experiment: US, safe risk level, not on giveaways blocklist.

In [ ]:
query = read_sql(
    "queries/baseline_conversion_rates.sql",
    params={"reference_date": "2026-03-15"},
)
conversion_df = bq.query(query).to_dataframe()

display_df = conversion_df.rename(columns={
    "segment": "Segment",
    "total_users": "Users",
    "conversions_7d": "Purchases (7d)",
    "conversions_10d": "Purchases (10d)",
    "conversions_14d": "Purchases (14d)",
    "conversion_rate_7d": "Rate (7d)",
    "conversion_rate_10d": "Rate (10d)",
    "conversion_rate_14d": "Rate (14d)",
})
for c in ["Rate (7d)", "Rate (10d)", "Rate (14d)"]:
    display_df[c] = display_df[c].apply(lambda v: f"{v:.2%}")

print("Observed Baseline Conversion Rates")
print("Reference date: 2026-03-15 · Users active in prior 30 days · US eligible\n")
display(display_df)

In [ ]:
CONVERSION_WINDOW = "14d"   # "7d", "10d", or "14d"
NUM_ARMS = 5                # treatment arms including control
MDE_PCT = 5                 # minimum detectable lift (% relative)
ALPHA = 0.05                # significance level
POWER = 0.80                # statistical power (1 - beta)

In [ ]:
rate_col = f"conversion_rate_{CONVERSION_WINDOW}"
comparisons = NUM_ARMS - 1
bonferroni_alpha = ALPHA / comparisons if comparisons > 1 else ALPHA
z_alpha = stats.norm.ppf(1 - bonferroni_alpha / 2)
z_beta = stats.norm.ppf(POWER)

overall = conversion_df[conversion_df["segment"] == "All users"].iloc[0]
p = float(overall[rate_col])
pool = int(overall["total_users"])
delta = p * (MDE_PCT / 100.0)
p_alt = p + delta

n_per_arm = int(np.ceil(
    (z_alpha * np.sqrt(2 * p * (1 - p)) + z_beta * np.sqrt(p * (1 - p) + p_alt * (1 - p_alt))) ** 2
    / delta ** 2
))
total_n = n_per_arm * NUM_ARMS
feasible = pool >= total_n

results = pd.DataFrame([{
    "Baseline Rate": f"{p:.2%}",
    "Detectable Lift": f"+{delta:.3%} (abs) / +{MDE_PCT:.0f}% (rel)",
    "Target Rate": f"{p_alt:.2%}",
    "N per Arm": f"{n_per_arm:,}",
    "Total N Required": f"{total_n:,}",
    "Eligible Pool": f"{pool:,}",
}])

verdict = "Feasible" if feasible else "Not feasible"
detail = (
    f"({pool:,} eligible users available)"
    if feasible
    else f"(only {pool:,} available — consider a larger MDE or fewer arms)"
)

display(results)
print(f"\n{verdict} — need {total_n:,} users across {NUM_ARMS} arms {detail}")
if comparisons > 1:
    print(f"Bonferroni-adjusted alpha = {bonferroni_alpha:.4f} for {comparisons} pairwise comparisons")

In [ ]:
curves = []
for m in range(1, 31):
    d = p * (m / 100.0)
    p_a = p + d
    n = int(np.ceil(
        (z_alpha * np.sqrt(2 * p * (1 - p)) + z_beta * np.sqrt(p * (1 - p) + p_a * (1 - p_a))) ** 2
        / d ** 2
    ))
    curves.append({"mde_pct": m, "n_per_arm": n, "total_n": n * NUM_ARMS})

curve_df = pd.DataFrame(curves)
max_per_arm = pool // NUM_ARMS

fig = px.line(
    curve_df,
    x="mde_pct",
    y="n_per_arm",
    title=f"How small a lift can we detect? ({CONVERSION_WINDOW} window, baseline = {p:.2%})",
    labels={"mde_pct": "Minimum Detectable Lift (% relative)", "n_per_arm": "N per Arm"},
    markers=True,
)
fig.data[0].name = "Required N per arm"
fig.data[0].showlegend = True
fig.add_scatter(
    x=[curve_df["mde_pct"].min(), curve_df["mde_pct"].max()],
    y=[max_per_arm, max_per_arm],
    mode="lines",
    line=dict(dash="dash", color="red"),
    name=f"Max available per arm ({max_per_arm:,})",
)
fig.update_layout(
    yaxis_type="log",
    yaxis_title="N per Arm (log scale)",
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.95),
)
fig.show()

min_feasible = curve_df[curve_df["n_per_arm"] <= max_per_arm]["mde_pct"].min()
print(f"With the current pool, the smallest detectable lift is ~{min_feasible}% relative.")
print("Below the red line = feasible; above = not enough users.")